# Práctica 4: Clustering

## Objetivos

El objetivo de esta práctica es utilizar el entorno **Google Colab** para afrontar un problema de **clustering**, aplicando distintos tipos de algoritmos, explorando la configuración de cada algoritmo, comprobando sus características y comparando sus resultados para evaluar su idoneidad en el problema presentado.

<br>
<br>



## Parte I: Descripción del problema

Disponemos de un fichero Excel denominado `online_shoppers_intention.xlsx` que contiene un conjunto de datos obtenidos de una web de comercio online, de la que se han recogido datos de 12.330 sesiones de usuarios reflejadas en 18 variables (categóricas y numéricas). Cada fila del conjunto de datos (instancia) pertenece a la sesión de un usuario diferente. Los datos recogidos corresponden a un período de un año para evitar cualquier tendencia a una campaña específica, un día especial, un perfil de usuario o un período.

Sobre este conjunto de datos, se pretende determinar la intención de los usuarios de un sitio web de finalizar una transacción de compra electrónica. Para ello, se aplicarán algoritmos de clustering para intentar descubrir grupos (clústeres) de usuarios con características similares que terminaron con una transacción para, potencialmente, segmentar estrategias de marketing o personalización. Esto se traduciría en ser capaces de ofrecer contenido sólo para aquellos usuarios que tienen la intención de comprar.

Hay que destacar que el conjunto de datos contiene una variable, **Ingresos** (*Revenue*), que indica si la sesión de usuario correspondiente terminó con una transacción o no (valores Verdadero o Falso). En particular, el 84,5% (10.422) de las sesiones no terminaron en compra (transacción), mientras que el resto (1.908) sí terminaron en compra. Esta variable **NO** debe utilizarse en el proceso de obtención de los clusteres, se debe excluir del proceso. Sólo se utilizará a posteriori para evaluar la bondad de los clústeres obtenidos como estrategia de segmentación de los usuarios.

<br>

### 1. Descripción de los atributos

El conjunto de datos consta de atributos numéricos y categóricos.

Los atributos numéricos son los siguientes:
* **Administrativo** *(Administrative)*: número de páginas visitadas por el visitante sobre la administración de cuentas.
* **Duración administrativa** *(Administrative_Duration)*: cantidad total de tiempo (en segundos) invertido por el visitante en las páginas relacionadas con la administración de cuentas
* **Informativo** *(Informational)*: número de páginas visitadas por el visitante sobre el sitio web, la comunicación y la dirección información del sitio de compras
* **Duración informativa** *(Informational_Duration)*: cantidad total de segundos empleados por el visitante en páginas informativas
* **Producto relacionado** *(ProductRelated)*: número de páginas visitadas sobre páginas relacionadas con el producto
* **Duración relacionada con el producto** *(ProductRelated_Duration)*: cantidad total de tiempo (en segundos) empleado por el visitante en páginas relacionadas con el producto
* **Índice de rebote** *(BounceRates)*: valor del porcentaje de rebote promedio de las páginas visitadas por el visitante
* **Tasa de salida** *(ExitRates)*: valor de tasa de salida promedio de las páginas visitadas por el visitante
* **Valor de página** *(PageValues)*: valor de página promedio de las páginas visitadas por el visitante
* **Día especial** *(SpecialDay)*: proximidad del sitio visitando un día especial

Los atributos categóricos se definen de la siguiente forma:
* **Sistema Operativo** *(OperatingSystems)*: sistema operativo del visitante
* **Navegador** *(Browser)*: navegador del visitante
* **Región** *(Region)*: región geográfica en la cual la sesión ha sido iniciada
* **Tipo de tráfico** *(TrafficType)*: fuente de tráfico por la cual el visitante ha llegado al sitio web (por ejemplo, banner, SMS, directo)
* **Tipo de visitante** *(VisitorType)*: “Nuevo", “Que regresa" y "Otro"
* **Fin de semana** *(Weekend)*: indica si la fecha es fin de semana
* **Mes** *(Month)*: número del mes de la fecha de visita
* **Ingresos** *(Revenue)*: indica si la visita se ha finalizado con una transacción

<br>

### 2. Problema a resolver

A partir de estos datos, se pretende obtener grupos (clústeres) de usuarios con características similares, que permita su segmentación para poder lanzar acciones sobre los usuarios que tengan intención de realizar una compra (terminar con una transacción).

Para afrontar el problema, utilizaremos los algoritmos de clustering disponibles en **Google Colab**. Recordar que el análisis debe excluir la variable **Ingresos** (*Revenue*), que sólo utilizaremos para evaluar los clústeres obtenidos.

También se debe tener en cuenta que se deben utilizar los conocimientos adquiridos sobre preprocesamiento para la preparación de los datos de cara a su uso por los distintos algoritmos de clustering.

El proceso completo incluye el preprocesamiento adecuado a la tarea de clustering a desarrollar, la práctica con distintos tipos de algoritmos de clustering y sus parámetros, y la comparación de los resultados de las distintas experimentaciones sobre algoritmos y sus parámetros.

<br>
<br>



## Parte II: Segmentación con K-Means

En esta sección se recoge el proceso completo para la aplicación de un algoritmo de clustering a este conjunto de datos. Así, describiremos la carga del conjunto de datos, el análisis exploratorio de estos datos, el preprocesamiento básico para trabajar con algoritmos de clustering, la aplicación del algoritmo K-Means y el análisis de sus resultados.

<br>

### 1. Carga del conjunto de datos

Primero importamos las librerías que necesitamos para todo el cuaderno:



In [ ]:
# Importar librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.stats import entropy

print("Librerías cargadas.")

Para cargar el conjunto de datos almacenado en el fichero excel `online_shoppers_intention.xlsx`, nos conectaremos con Google Drive e indicaremos la ruta del fichero en nuestro directorio:


In [ ]:
# Montar Google Drive para acceder al dataset
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive montado correctamente.")

# Reemplaza 'AQUI_EL_PATH_DE_TU_FICHERO' con la ruta real de tu dataset en Google Drive
file_path = '/content/drive/MyDrive/AQUI_EL_PATH_DE_TU_FICHERO/online_shoppers_intention.xlsx'
try:
    df = pd.read_excel(file_path)
    print("Dataset cargado con éxito!")
    print(f"Dimensiones del dataset: {df.shape}")
    display(df.head())
except FileNotFoundError:
    print(f"Error: No se encuentra el archivo en la ruta {file_path}. Por favor, verifica la ruta.")
except Exception as e:
    print(f"Ocurrió un error al cargar el dataset: {e}")

<br>

### 2. Análisis exploratorio de datos

Debemos conocer los datos con los que vamos a trabajar, por lo que desarrollaremos un anális exploratorio de datos **básico**:
* Manejo de valores nulos
* Análisis de variables categóricas
* Análisis de variables numéricas

#### 2.1. Manejo de valores nulos


In [ ]:
print('--- Verificación de valores nulos ---')
missing_values = df.isnull().sum()

if missing_values.sum() == 0:
    print("No hay valores nulos en el dataset.")
else:
    print("Valores nulos por columna:")
    print(missing_values[missing_values > 0])

En este caso, el dataset no contiene valores nulos, por lo que no se requiere su tratamiento mediante imputación o eliminación.

#### 2.2 Análisis de variables categóricas

In [ ]:
print('--- Identificacion de variables categóricas ---')
categorical_cols = df.select_dtypes(include=['object', 'bool']).columns.tolist()
print(f"Columnas categóricas identificadas: {categorical_cols}")

print('\n--- Análisis de la distribución de variables categóricas ---')
for col in categorical_cols:

    plt.figure(figsize=(16, 6))

    # Las figuras se muestran en en dos columnas

    # Subplot 1: Percentage Distribution
    plt.subplot(1, 2, 1)
    percentage_dist = df[col].value_counts(normalize=True).mul(100).sort_values(ascending=False)
    # Convert index to string to ensure 'True'/'False' are plotted correctly instead of 1/0
    sns.barplot(x=percentage_dist.values, y=percentage_dist.index.astype(str))
    plt.title(f'Distribución porcentual de {col}')
    plt.xlabel('Porcentaje (%)')
    plt.ylabel('') # Remove y-label as it's already in the title

    # Subplot 2: Conteo de valores
    plt.subplot(1, 2, 2)
    value_counts_series = df[col].value_counts()
    text_output = ""
    for index, value in value_counts_series.items():
        text_output += f"{index}: {value}\n"
    plt.text(0.05, 0.95, text_output, transform=plt.gca().transAxes,
             fontsize=14, verticalalignment='top', bbox=dict(boxstyle="round,pad=0.3", fc="lightgray", ec="gray", lw=0.5))
    plt.gca().set_axis_off() # Ocultar ejes
    plt.title(f'Conteo de {col}')

    plt.tight_layout()
    plt.show()

#### 2.3 Análisis de variables numéricas



In [ ]:
print('--- Identificación de variables numéricas ---')
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Columnas numéricas identificadas: {numerical_cols}")

print('\n--- Estadísticas descriptivas de las variables numéricas ---')
display(df[numerical_cols].describe())

print('\n--- Visualización de la distribución de las variables numéricas (Histogramas) ---')
num_cols_to_plot = numerical_cols[:len(numerical_cols)]
filas = int(len(numerical_cols)/3)+1
plt.figure(figsize=(15, 10))
for i, col in enumerate(num_cols_to_plot):
    plt.subplot(filas, 3, i + 1) # Disposición a 3 columnas
    sns.histplot(df[col], kde=True)
    plt.title(f'Distribución de {col}')
    plt.xlabel('')
    plt.ylabel('')
plt.tight_layout()
plt.show()

### 3. Preprocesamiento

Basándonos en el análisis exploratorio de datos previo, vamos a desarrollar aquí un preprocesamiento muy básico, que deberá ser mejorado y completado en el desarrollo de la práctica:

*   **Eliminación de la variable 'Revenue'**: la variable 'Revenue' (ingresos) debe excluirse para el clustering, porque contiene la información sobre si la visita ha finalizado con una transacción.

*   **Codificación de Variables Categóricas**: Las columnas como `Month`, `VisitorType` y `Weekend` son de tipo `object` o `bool` y representan categorías. Los algoritmos de clustering, en su mayoría, requieren entradas numéricas. Se aplicará **One-Hot Encoding** para convertir estas variables en un formato numérico.

*   **Escalado de Variables Numéricas**: Las variables numéricas presentan rangos de valores muy diferentes, como se ve en las estadísticas descriptivas y los histogramas. Por ejemplo, `Administrative` y `ProductRelated` tienen valores máximos significativamente más altos que `BounceRates` o `SpecialDay`. Los algoritmos de clustering que se basan en la distancia (como K-Means o Agglomerative Clustering) son sensibles a la escala de las características. Para evitar que las características con rangos más grandes dominen el cálculo de la distancia, se van a escalar estas variables, utilzando en este caso `StandardScaler`, para que tengan una media de 0 y una varianza de 1. Esto asegurará que ningún atributo domine indebidamente el proceso de clustering debido a su escala, permitiendo que los algoritmos encuentren patrones basados en la estructura real de los datos.


In [ ]:
print('--- Identificación de variables categóricas y numéricas ---')

# Eliminamos la variable 'Revenue' (ingresos), porque contiene la información sobre si la visita ha finalizado con una transacción. Las demás columnas son las características.
features_df = df.drop(columns=['Revenue'])

categorical_cols = features_df.select_dtypes(include=['object', 'bool']).columns.tolist()
print(f"Columnas categóricas: {categorical_cols}")

numerical_cols = features_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Columnas numéricas: {numerical_cols}")


In [ ]:
print('--- Aplicación de One-Hot Encoding y Escalado ---')

# Como hemos comentado previamente, la columna 'Revenue' no debe usarse para el clustering.
x = df.drop('Revenue', axis=1)

# 'Weekend' es boleana y es tratada como numérica binaria por ColumnTransformer, pero pueden ser pasadas a OneHotEncoder si se desea una columna para True y otra para False.
# Para este caso, dado que 'Weekend' ya es True/False (0/1), se tratará bien con StandardScaler o se dejará tal cual si no se escala.
# Por lo tanto, la incluimos en las columnas numéricas que serán escaladas.
# 'Month' es categórica y requiere ser codificada con OneHotEncoder.
# 'VisitorType' es categórica y requiere OHE.
# 'OperatingSystems', 'Browser', 'Region', 'TrafficType' son categóricas y requieren OHE.

categorical_cols_processed = x.select_dtypes(include=['object']).columns.tolist()

numerical_cols_processed = x.select_dtypes(include=['int64', 'float64', 'bool']).columns.tolist()

# Asegurarse de que 'Weekend' no esté en categóricas si ya está en numéricas
if 'Weekend' in categorical_cols_processed:
    categorical_cols_processed.remove('Weekend')

# Se aplica StandardScaler a las variables numéricas y OneHotEncoder a las categóricas
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols_processed),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols_processed)
    ])

# Crear un pipeline para el preprocesamiento
pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

# Aplicar el preprocesamiento a los datos
X_processed = pipeline.fit_transform(features_df)

# Obtener los nombres de las nuevas columnas para el DataFrame procesado
feature_names = numerical_cols_processed + list(pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_cols_processed))

# Convertir la matriz dispersa resultante en un DataFrame de Pandas
X_processed_df = pd.DataFrame(X_processed, columns=feature_names)

print(f"Dimensiones del dataset preprocesado: {X_processed_df.shape}")
display(X_processed_df.head())

print('\nPreprocesamiento completado. El dataset está listo para clustering.')


### 4. Obtención de clusters con K-Means

A continuación se describe cómo aplicar el algoritmo de clustering **K-Means** sobre el conjunto de datos preprocesados. K-Means es un algoritmo de clustering particional que busca dividir `n` observaciones en `k` clústeres, donde cada observación pertenece al clúster cuyo centroide le es más cercano.

Puesto que K-Means requiere fijar a priori el número de clústeres a obtener, primero vamos a determinar el número óptimo de clústeres utilizando el método del codo (que utiliza SSE) y el coeficiente de silueta. Esto requiere la ejecución del algoritmo con distintos valores de `k`.


In [ ]:
print('--- Determinación del número óptimo de clústeres para K-Means (Método del Codo y Silhouette Score) ---\n')

# Usaremos un rango de clústeres para evaluar, en este caso de 2 a 10 clústeres
k_range = range(2, 11)

inertia = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10) # n_init para asegurar convergencia
    kmeans.fit(X_processed_df)
    inertia.append(kmeans.inertia_)
    score = silhouette_score(X_processed_df, kmeans.labels_)
    silhouette_scores.append(score)

# Gráfico del Método del Codo
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(k_range, inertia, marker='o')
plt.title('Método del Codo (Inertia)')
plt.xlabel('Número de Clústeres (k)')
plt.ylabel('Inertia')
plt.xticks(list(k_range))
plt.grid(True)

# Gráfico del Silhouette Score
plt.subplot(1, 2, 2)
plt.plot(k_range, silhouette_scores, marker='o')
plt.title('Silhouette Score')
plt.xlabel('Número de Clústeres (k)')
plt.ylabel('Score')
plt.xticks(list(k_range))
plt.grid(True)

plt.tight_layout()
plt.show()

print('\nAnálisis de los gráficos:')
print('El Método del Codo busca un "codo" en la curva de inertia, donde la mejora disminuye.')
print('El Coeficiente de Silueta (valores de -1 a 1) busca el valor más alto, indicando clústeres bien separados y densos.')

Basado en el análisis de los gráficos anteriores, debemos elegir un número de clústeres. En este ejemplo vamos a elegir 3 clústeres, pero se debe experimentar con distintos valores para comprobar con cuál de ellos se obtienen mejores resultados.



In [ ]:
print('\n--- Aplicación de K-Means con el número de clústeres elegido ---')

# Para este ejemplo, vamos a elegir 3 clústeres, pero se debe ajustar este valor y probar con distintas opciones.
n_clusters_kmeans = 3
kmeans_model = KMeans(n_clusters=n_clusters_kmeans, random_state=42, n_init=10) # Volvemos a ejecutar el algorimto, fijando ya 3 clústeres
kmeans_labels = kmeans_model.fit_predict(X_processed_df)

print(f"K-Means aplicado con {n_clusters_kmeans} clústeres.")
print(f"Distribución de clústeres:\n{pd.Series(kmeans_labels).value_counts()}")

# Evaluación adicional
sil_score = silhouette_score(X_processed_df, kmeans_labels)
print(f"Silhouette Score para K-Means con {n_clusters_kmeans} clústeres: {sil_score:.4f}")

El resultado del clustering se puede visualizar reduciendo el gráfico a 2 dimensiones utilizando PCA (Análisis de Componentes Principales):

In [ ]:
print('\n--- Visualización de los resultados de K-Means ---\n')

# Realizamos una reducción de dimensionalidad con PCA para visualizar el resultado en 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_processed_df)

pca_df = pd.DataFrame(data=X_pca, columns=['Principal Component 1', 'Principal Component 2'])
pca_df['Cluster_KMeans'] = kmeans_labels

plt.figure(figsize=(10, 8))
sns.scatterplot(x='Principal Component 1', y='Principal Component 2', hue='Cluster_KMeans', data=pca_df, palette='viridis', legend='full', alpha=0.7)
plt.title(f'Visualización de Clústeres K-Means (k={n_clusters_kmeans}) con PCA')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.grid(True)
plt.show()

print('\nAnálisis de la visualización:')
print('Observa la separación y compactitud de los clústeres. Idealmente, los puntos dentro de un clúster deberían estar cerca, y los clústeres deberían estar bien separados entre sí.')

Pero podemos mejorar la visualización del resultado analizando la **pureza** de los clústeres respecto a `Revenue`: para evaluar la calidad de los clústeres en relación con la variable `Revenue` (que indica si la visita terminó en compra), podemos calcular la entropía o simplemente ver la distribución de `Revenue` dentro de cada clúster. Un clúster 'puro' tendría una alta proporción de 0s o 1s en `Revenue`.


In [ ]:
# Crear un DataFrame temporal para el análisis de pureza
df_temp_analysis = pd.DataFrame({'Revenue': df['Revenue'], 'KMeans_Cluster': kmeans_labels})

cluster_revenue_distribution = df_temp_analysis.groupby('KMeans_Cluster')['Revenue'].value_counts(normalize=True).unstack(fill_value=0)
print("Distribución de 'Revenue' por clúster (K-Means):")
display(cluster_revenue_distribution)

# Calcular entropía para cada clúster
entropy_scores_kmeans = {}
for cluster_id in cluster_revenue_distribution.index:
    probs = cluster_revenue_distribution.loc[cluster_id].values
    # Asegurarse de que no haya probabilidades cero para evitar log(0)
    probs = probs[probs > 0]
    entropy_scores_kmeans[cluster_id] = entropy(probs, base=2)

print("\nEntropía de 'Revenue' por clúster (K-Means):")
for cluster_id, ent_score in entropy_scores_kmeans.items():
    print(f"Clúster {cluster_id}: {ent_score:.4f}")


# Visualización de clústeres (primeros dos componentes principales si son más de 2 dimensiones)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_processed_df)

pca_df = pd.DataFrame(data=X_pca, columns=['Principal Component 1', 'Principal Component 2'])
pca_df['Cluster_KMeans'] = kmeans_labels
pca_df['Revenue'] = df['Revenue'].values # Añadir la columna Revenue al DataFrame PCA para la visualización

plt.figure(figsize=(12, 10))
sns.scatterplot(x='Principal Component 1', y='Principal Component 2',
                hue='Cluster_KMeans', style='Revenue',
                data=pca_df, palette='viridis', legend='full', s=50, alpha=0.7)
plt.title(f'Visualización de Clústeres K-Means (k={n_clusters_kmeans}) con PCA y Revenue')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.grid(True)
plt.show()

<br><br>

## Parte III: Trabajo a realizar

La práctica se realiza de forma individual, creando un nuevo cuaderno de Google Colab partiendo de éste, que contenga los siguientes elementos:

1. Carga y preprocesamiento mejorado de los datos para adecuarlos a los algoritmos de clustering.
2. Aplicación del algoritmo K-Means sobre los nuevos datos preprocesados.
3. Aplicación de otros algoritmos de clustering.
4. Comparación de los resultados de los distintos algoritmos.
5. Análisis de los resultados y conclusiones.

El contenido a desarrollar en cada uno de estos puntos se detalla a continuación.

No hay que entregar una memoria en PDF, sino que la memoria será el propio cuaderno desarrollado. Por lo tanto es necesario ir explicando el trabajo que se vaya realizando, las decisiones que se tomen y los resultados que se obtengan en las celdas de texto y en comentarios del código del propio cuaderno. Sólo hay que subir el cuaderno desarrollado, en un fichero  `.ipyng`, a la actividad correspondiente a la práctica 4 del módulo I en PLATEA.


### 1. Carga y preprocesamiento de los datos

En primer lugar, se debe mejorar y completar el preprocesamiento de los datos incluido en la segunda parte de este cuaderno utilizando los conocimientos adquiridos de análisis exploratorio de datos. Se deben realizar tareas de preprocesamiento que adecúen los datos a las necesidades y características de los algoritmos de clústering, lo que incluye entre otras:
* Utilizar aquellas variables que caracterizan al usuario.
* Ignorar aquellas variables que no se consideren necesarias para el estudio descriptivo del problema.
* Aplicar normalización para que las métricas de distancia y visualización funcionen correctamente.
* Comprobar que todas las variables sean numéricas.


### 2. Aplicación del algoritmo K-Means

A continuación, se volverá a utilizar el algoritmo K-Means pero aplicándolo a los nuevos datos preprocesados, experimentando con los parámetros del algoritmo, para finalmente comparar los resultados con los obtenidos en la segunda parte de este cuaderno. Se deberá comentar el proceso seguido para la experimentación, los parámetros utilizados y los posibles problemas encontrados durante la experimentación.


### 3. Aplicación de otros algoritmos de clustering

El siguiente paso será la ejecución de nuevos algoritmos de clustering sobre los datos preprocesados. En particular, se debe utilizar al menos **UN algoritmo de cada uno de los siguientes tipos**:
* Jerárquico.
* Basado en densidad.
* Basado en rejillas o basado en modelos.

De la misma forma que para K-means, se debe experimentar con distntos valores de los parámetros de cada algoritmo, buscando aquellas combinaciones que permitan obtener mejores resultados en este problema. Se visualizarán los resultados de cada algoritmo con los gráficos que se consideran adecuados. Tambén se incluirán comentarios sobre la experimentación, como posibles problemas encontrados para determinar los parámetros, etc.


### 4. Comparación de resultados

Se debe realizar una comparativa de los resultados obtenidos por los distintos algoritmos, utilizando métricas como la suma de errores cuadráticos (SSE), el coeficiente de silueta y la entropía de los clústeres con respecto a la variable 'Revenue' (pureza de los clusteres). Esta comparativa se completará con una breve descripción de los resultados, incluyendo las gráficas necesarias para una mejor explicación de los mismos.


### 5. Análisis de los resultados y conclusiones

Por último, se deben desarrollar unas conclusiones, centradas en el análsis de cuál de los algoritmos parece ser el más adecuado para este problema concreto y por qué.

